# Run the Complete Retail Demand Forecasting Project

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmavigo/retail-demand-forecasting/blob/main/notebooks/00_RUN_COMPLETE_PROJECT_IN_COLAB.ipynb)

This is the easiest way to reproduce the complete project. Run each cell from top to bottom. Colab provides the Python environment; Kaggle provides the competition data.

## 1. Prepare Google Colab
This cell downloads the repository and installs the required packages. It may take several minutes the first time.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL='https://github.com/ericmavigo/retail-demand-forecasting.git'
ROOT=Path('/content/retail-demand-forecasting') if 'google.colab' in sys.modules else (Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd())
if 'google.colab' in sys.modules and not ROOT.exists():
    subprocess.run(['git','clone','--depth','1',REPO_URL,str(ROOT)],check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements-dev.txt')],check=True)
print('Project root:',ROOT)

## 2. Connect Kaggle
Accept the M5 competition rules first. Kaggle will request your API token. The token stays in the Colab session and is never committed to GitHub.

In [ ]:
import kagglehub
if 'google.colab' in sys.modules:
    kagglehub.login()
print('Kaggle authentication ready')

## 3. Download directly with KaggleHub
`competition_download` returns the local folder containing the latest authorized M5 files. The checks below make the data location explicit and stop immediately if a required table is missing.

In [ ]:
RAW=ROOT/'data'/'raw'; PROCESSED=ROOT/'data'/'processed'; REPORTS=ROOT/'reports'
RAW.mkdir(parents=True,exist_ok=True)

path=Path(kagglehub.competition_download(
    'm5-forecasting-accuracy',
    output_dir=str(RAW)
))
DATA_DIR=path if path.is_dir() and (path/'calendar.csv').exists() else RAW
if not (DATA_DIR/'calendar.csv').exists():
    DATA_DIR=next(p.parent for p in RAW.rglob('calendar.csv'))

required={'calendar.csv','sell_prices.csv','sales_train_evaluation.csv'}
missing=required-{p.name for p in DATA_DIR.glob('*.csv')}
assert not missing,f'Missing files: {sorted(missing)}'
print('Path to competition files:',DATA_DIR)
subprocess.run([sys.executable,str(ROOT/'src/data_audit.py'),'--data-dir',str(DATA_DIR),'--output-dir',str(REPORTS)],check=True)

## 4. Load and integrate the three core tables
The keys form a simple relational model: `sales.d` joins to `calendar.d`; then `store_id + item_id + wm_yr_wk` joins to prices. The preview uses 100 products so the relationship is easy to inspect without creating a 59-million-row DataFrame in memory.

In [ ]:
import json, numpy as np, pandas as pd, plotly.express as px

sales=pd.read_csv(DATA_DIR/'sales_train_evaluation.csv')
calendar=pd.read_csv(DATA_DIR/'calendar.csv',parse_dates=['date'])
prices=pd.read_csv(DATA_DIR/'sell_prices.csv')
day_cols=[c for c in sales.columns if c.startswith('d_')]
id_cols=['item_id','dept_id','cat_id','store_id','state_id']

integrated_preview=(
    sales.loc[:99,id_cols+day_cols].melt(
        id_vars=id_cols,var_name='d',value_name='units'
    )
    .merge(calendar,on='d',how='left',validate='many_to_one')
    .merge(
        prices,on=['store_id','item_id','wm_yr_wk'],
        how='left',validate='many_to_one'
    )
)
integrated_preview['estimated_revenue']=integrated_preview['units']*integrated_preview['sell_price']
display(integrated_preview[['date','store_id','item_id','cat_id','units','sell_price','estimated_revenue']].head())
print(f'Sales: {sales.shape} | Calendar: {calendar.shape} | Prices: {prices.shape}')

### Build the complete analytical tables
The production step applies the same keys to all 30,490 series in memory-efficient blocks, then saves daily, weekly, store, category and product outputs.

In [ ]:
subprocess.run([sys.executable,str(ROOT/'src/build_analytics.py'),'--data-dir',str(DATA_DIR),'--output-dir',str(PROCESSED)],check=True)
validation=json.loads((PROCESSED/'validation.json').read_text())
pd.Series(validation,name='value').to_frame()

## 5. Explore five years of performance

In [ ]:
daily=pd.read_csv(PROCESSED/'daily_overview.csv',parse_dates=['date'])
weekly=pd.read_csv(PROCESSED/'weekly_overview.csv',parse_dates=['date'])
stores=pd.read_csv(PROCESSED/'store_summary.csv')
products=pd.read_csv(PROCESSED/'product_summary.csv')
px.line(daily,x='date',y=['units','moving_average_28'],title='Five-year demand and 28-day trend').show()
px.bar(stores.sort_values('estimated_revenue'),x='estimated_revenue',y='store_id',orientation='h',title='Estimated revenue by store').show()

In [ ]:
p=products.copy(); p['product_share']=(p.index+1)/len(p)
px.line(p,x='product_share',y='cumulative_revenue_share',title='Product revenue Pareto curve').show()
products.head(20)

## 6. Train and evaluate LightGBM
The last 28 days remain unseen until evaluation. Training can take several minutes in Colab.

In [ ]:
subprocess.run([sys.executable,str(ROOT/'src/train_model.py'),'--data-dir',str(DATA_DIR),'--output-dir',str(PROCESSED),'--reports-dir',str(REPORTS),'--samples','800000'],check=True)

In [ ]:
metrics=pd.read_csv(REPORTS/'model_metrics.csv').sort_values('WAPE')
display(metrics.style.format({'MAE':'{:.4f}','WAPE':'{:.2%}','RMSSE':'{:.4f}','Bias':'{:.2%}'}))
px.bar(metrics.assign(WAPE_percent=100*metrics.WAPE).sort_values('WAPE_percent',ascending=False),x='WAPE_percent',y='model',orientation='h',text_auto='.2f',title='28-day holdout WAPE').show()

## 7. Review inventory decisions

In [ ]:
inventory=pd.read_csv(PROCESSED/'inventory_summary.csv')
display(inventory)
px.bar(inventory.melt('method',var_name='outcome',value_name='units'),x='method',y='units',color='outcome',barmode='group',title='Inventory trade-off').show()
reduction=1-inventory.iloc[1].stockout_units/inventory.iloc[0].stockout_units
print(f'Estimated stockout-unit reduction: {reduction:.1%}')

## Result
You have reproduced the data audit, cleaning, executive analysis, seasonality, global forecasting model and inventory simulator. Continue with the specialized notebooks to modify individual stages.